# Sovereign Debt Distress — Data Preprocessing Pipeline v2
## Changes from original:
| What | Before | After |
|------|--------|-------|
| ACLED political features | 0 (not merged!) | 18 (12 raw + 6 rolling) |
| BOP features | 11 raw + 4 derived | 15 raw + 4 derived + 4 rolling = 17 |
| BIS features | 0 | 1 (forward-filled quarterly) |
| Cross-feature interactions | 2 | 8 |
| Total feature columns | ~44 | 86 |
| Modalities covered | 2 (macro+debt) | 5 (macro+debt+political+bop+interactions) |


In [ ]:
import pandas as pd, numpy as np, json, os, warnings
warnings.filterwarnings('ignore')

panel = pd.read_csv("data/processed/panel_full.csv", parse_dates=["year_month"])
acled = pd.read_csv("data/interim/acled_country_month.csv", parse_dates=["year_month"])
bis = pd.read_csv("data/interim/bis_debt_securities_monthly.csv", parse_dates=["year_month"])
print(f"Panel: {panel.shape} | ACLED: {acled.shape} | BIS: {bis.shape}")
print(f"Labels y=1: {(panel['distress_within_12m']==1).sum()}")


## 1. Drop old missing flags, Merge ACLED + BIS


In [ ]:
# Drop old missing flags
old_miss = [c for c in panel.columns if c.endswith('_missing')]
panel = panel.drop(columns=old_miss, errors='ignore')

# Merge ACLED
acled_feats = [c for c in acled.columns if c not in ['iso3','year_month']]
panel = panel.merge(acled, on=['iso3','year_month'], how='left')
for col in acled_feats:
    panel[col] = panel[col].fillna(0)
print(f"ACLED: +{len(acled_feats)} features, {(panel['acled_total_events']>0).sum()} rows with events")

# Merge BIS
if 'bis_govt_debt_securities_outstanding' not in panel.columns:
    panel = panel.merge(bis, on=['iso3','year_month'], how='left')
    panel = panel.sort_values(['iso3','year_month'])
    panel['bis_govt_debt_securities_outstanding'] = panel.groupby('iso3')['bis_govt_debt_securities_outstanding'].ffill()
print(f"Panel now: {panel.shape[1]} columns")


## 2. Engineer ACLED + BOP rolling features


In [ ]:
panel = panel.sort_values(['iso3','year_month']).reset_index(drop=True)
for iso3, grp in panel.groupby('iso3'):
    idx = grp.index
    # ACLED rolling
    panel.loc[idx, 'acled_events_3m_avg'] = grp['acled_total_events'].rolling(3,min_periods=1).mean().values
    panel.loc[idx, 'acled_events_6m_avg'] = grp['acled_total_events'].rolling(6,min_periods=1).mean().values
    panel.loc[idx, 'acled_fatalities_3m_avg'] = grp['acled_total_fatalities'].rolling(3,min_periods=1).mean().values
    panel.loc[idx, 'acled_fatalities_6m_avg'] = grp['acled_total_fatalities'].rolling(6,min_periods=1).mean().values
    e3 = grp['acled_total_events'].rolling(3,min_periods=1).mean()
    panel.loc[idx, 'acled_protest_trend_3m'] = (grp['acled_protests'].rolling(3,min_periods=1).mean() - grp['acled_protests'].rolling(3,min_periods=1).mean().shift(3)).values
    panel.loc[idx, 'acled_events_momentum'] = (e3 - e3.shift(3)).values
    # BOP rolling
    panel.loc[idx, 'bop_reserves_6m_pct_change'] = grp['bop_reserve_assets_usd'].pct_change(6).values
    panel.loc[idx, 'bop_ca_6m_change'] = grp['bop_current_account_balance_usd'].diff(6).values
    panel.loc[idx, 'bop_fdi_3m_avg'] = grp['bop_fdi_liabilities_usd'].rolling(3,min_periods=1).mean().values
    panel.loc[idx, 'bop_portfolio_volatility_6m'] = grp['bop_portfolio_liabilities_usd'].rolling(6,min_periods=1).std().values
print("Added 6 ACLED + 4 BOP rolling features")


## 3. Cross-feature interactions


In [ ]:
panel['instability_x_debt'] = panel['acled_total_events'] * panel['edt_debt_to_gni'].fillna(0) / 100
panel['fatalities_x_fx_stress'] = panel['acled_total_fatalities'] * panel['fx_3m_depreciation'].fillna(0).abs()
panel['violence_per_gdp'] = (panel['acled_total_events'] / panel['weo_gdp_per_capita_ppp'].replace(0,np.nan)).fillna(0)
for iso3, grp in panel.groupby('iso3'):
    panel.loc[grp.index, 'debt_to_gni_6m_change'] = grp['edt_debt_to_gni'].diff(6).values
panel['short_term_debt_pct'] = panel['edt_short_term_debt'] / panel['edt_total_external_debt'].replace(0,np.nan) * 100
if 'bop_reserves_to_external_debt_ratio' not in panel.columns:
    panel['bop_reserves_to_external_debt_ratio'] = panel['bop_reserve_assets_usd'] / panel['edt_total_external_debt'].replace(0,np.nan)
panel['capital_flow_stress'] = -panel['bop_portfolio_liabilities_usd'].fillna(0) - panel['bop_reserve_assets_usd_3m_change'].fillna(0)
print("Added 7 interaction features")


## 4. Imputation, splits, save


In [ ]:
id_cols = ['iso3','year_month','country_name','year','month_num']
label_cols = ['distress_within_12m','in_distress_now','months_to_next_distress']
feature_cols = [c for c in panel.columns if c not in id_cols+label_cols and not c.endswith('_missing') and c!='split' and pd.api.types.is_numeric_dtype(panel[c])]

# Missing flags
high_miss = [c for c in feature_cols if panel[c].isna().mean() > 0.10]
mf = []
for col in high_miss:
    fn = f"{col}_missing"; panel[fn] = panel[col].isna().astype(int); mf.append(fn)

# Impute
panel = panel.sort_values(['iso3','year_month']).reset_index(drop=True)
for col in feature_cols:
    panel[col] = panel.groupby('iso3')[col].ffill()
    panel[col] = panel.groupby('iso3')[col].bfill(limit=6)
    if panel[col].isna().any(): panel[col] = panel[col].fillna(panel[col].median())

# Splits
panel['split'] = 'train'
panel.loc[panel['year_month']>='2017-01-01','split'] = 'val'
panel.loc[panel['year_month']>='2020-01-01','split'] = 'test'

os.makedirs("data/final", exist_ok=True)
flat_features = feature_cols + mf
for s in ['train','val','test']:
    sub = panel[panel['split']==s]
    sub[['iso3','year_month']+flat_features+['distress_within_12m']].to_csv(f"data/final/{s}_flat.csv",index=False)
    y1=(sub['distress_within_12m']==1).sum()
    print(f"{s:5s}: {len(sub)} rows | y=1: {y1} ({y1/len(sub)*100:.1f}%) | features: {len(flat_features)}")

# Windows
WINDOW=24
all_X,all_y,all_meta=[],[],[]
for iso3,grp in panel.sort_values(['iso3','year_month']).groupby('iso3'):
    grp=grp.reset_index(drop=True); vals=grp[feature_cols].values.astype(np.float32)
    labs=grp['distress_within_12m'].fillna(0).astype(int).values; splits=grp['split'].values; months=grp['year_month'].values
    for end in range(WINDOW-1,len(grp)):
        all_X.append(vals[end-WINDOW+1:end+1]); all_y.append(int(labs[end]))
        all_meta.append({'iso3':iso3,'window_end':str(months[end])[:10],'y':int(labs[end]),'split':splits[end]})
all_X,all_y=np.stack(all_X),np.array(all_y); meta_df=pd.DataFrame(all_meta)
for s in ['train','val','test']:
    mask=meta_df['split']==s
    np.savez_compressed(f"data/final/{s}_windows.npz",X=all_X[mask.values],y=all_y[mask.values])
    meta_df[mask].to_csv(f"data/final/{s}_meta.csv",index=False)
    print(f"{s} windows: X={all_X[mask.values].shape} | y=1: {(all_y[mask.values]==1).sum()}")

panel.to_csv("data/final/panel_full.csv",index=False)
modality_map={"macro":[c for c in feature_cols if any(c.startswith(p) for p in ['cpi_','fx_','weo_','ir_'])],"debt":[c for c in feature_cols if any(c.startswith(p) for p in ['edt_','bis_','debt_stress','short_term_debt_pct','debt_to_gni_6m'])],"political":[c for c in feature_cols if any(c.startswith(p) for p in ['acled_','political_violence','instability_x','fatalities_x','violence_per'])],"bop":[c for c in feature_cols if c.startswith('bop_')],"interactions":['capital_flow_stress']}
json.dump({"feature_columns":feature_cols,"missing_flags":mf,"flat_features":flat_features,"modality_map":modality_map,"window_size":WINDOW,"n_features":len(feature_cols)},open("data/final/feature_metadata.json","w"),indent=2)
print(f"\nFINAL: {len(feature_cols)} features across {len([k for k,v in modality_map.items() if v])} modalities")
